In [1]:
import pandas as pd

df = pd.read_csv('train2.csv')

print(df.shape)      # 查看行列数
print(df.columns)    # 查看列名
df.head()

(364013, 45)
Index(['loanAmnt', 'term', 'interestRate', 'installment', 'grade', 'subGrade',
       'employmentTitle', 'employmentLength', 'homeOwnership', 'annualIncome',
       'verificationStatus', 'isDefault', 'purpose', 'postCode', 'regionCode',
       'dti', 'delinquency_2years', 'ficoRangeLow', 'ficoRangeHigh', 'openAcc',
       'pubRec', 'pubRecBankruptcies', 'revolBal', 'revolUtil', 'totalAcc',
       'initialListStatus', 'applicationType', 'earliesCreditLine', 'title',
       'n0', 'n1', 'n2', 'n3', 'n4', 'n5', 'n6', 'n7', 'n8', 'n9', 'n10',
       'n11', 'n12', 'n13', 'n14', 'issueDateDT'],
      dtype='object')


,loanAmnt,term,interestRate,installment,grade,subGrade,employmentTitle,employmentLength,homeOwnership,annualIncome,...,n6,n7,n8,n9,n10,n11,n12,n13,n14,issueDateDT
0,35000.0,5,19.52,917.97,4,21,320.0,2.0,2,110000.0,...,8.0,4.0,12.0,2.0,7.0,0.0,0.0,0.0,2.0,2587.0
1,18000.0,5,18.49,461.90,3,16,219843.0,5.0,0,46000.0,...,4.0,6.0,11.0,4.0,13.0,0.0,0.0,0.0,1.0,1888.0
2,12000.0,5,16.99,298.17,3,17,31698.0,8.0,0,74000.0,...,21.0,4.0,5.0,3.0,11.0,0.0,0.0,0.0,4.0,3044.0
3,2050.0,3,7.69,63.95,0,3,180083.0,9.0,0,35000.0,...,3.0,10.0,18.0,3.0,12.0,0.0,0.0,0.0,3.0,2679.0
4,11500.0,3,14.98,398.54,2,12,214017.0,1.0,1,30000.0,...,10.0,5.0,21.0,4.0,8.0,0.0,0.0,0.0,2.0,2406.0


In [3]:
from sklearn.model_selection import train_test_split

X = df.drop('isDefault',axis=1)
y = df['isDefault']

# 测试集10%
X_temp,X_test,y_temp,y_test = train_test_split(
    X,
    y,
    test_size=0.1,
    random_state=42,
    stratify=y
)

# 验证集20%
X_train,X_val,y_train,y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=2/9,
    random_state=42,
    stratify=y_temp
)

print("训练集:",X_train.shape)
print("验证集:",X_val.shape)
print("测试集:",X_test.shape)

训练集: (254808, 44)
验证集: (72803, 44)
测试集: (36402, 44)


In [5]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

def objective_xgb(trial):

    params = {

        'learning_rate':
        trial.suggest_float(
            'learning_rate',
            0.01,
            0.1
        ),

        'max_depth':
        trial.suggest_int(
            'max_depth',
            3,
            7
        ),

        'n_estimators':
        trial.suggest_int(
            'n_estimators',
            100,
            500
        ),

        'min_child_weight':
        trial.suggest_int(
            'min_child_weight',
            1,
            10
        ),

        'subsample':
        trial.suggest_float(
            'subsample',
            0.6,
            1.0
        ),

        'colsample_bytree':
        trial.suggest_float(
            'colsample_bytree',
            0.6,
            1.0
        ),

        'eval_metric':'auc',

        'random_state':42
    }

    model = XGBClassifier(**params)

    model.fit(X_train,y_train)

    pred = model.predict_proba(X_val)[:,1]

    auc = roc_auc_score(y_val,pred)

    return auc

study_xgb = optuna.create_study(
    direction='maximize'
)

study_xgb.optimize(
    objective_xgb,
    n_trials=50
)

print("最佳AUC")
print(study_xgb.best_value)

print("\n最佳参数")
print(study_xgb.best_params)

[I 2026-05-21 08:01:24,870] A new study created in memory with name: no-name-6a29f950-5bd1-4e5f-afa2-684e7ae317be
[I 2026-05-21 08:02:31,348] Trial 0 finished with value: 0.7235306438454154 and parameters: {'learning_rate': 0.01672820594863468, 'max_depth': 5, 'n_estimators': 368, 'min_child_weight': 1, 'subsample': 0.6852444325655698, 'colsample_bytree': 0.9717883111040284}. Best is trial 0 with value: 0.7235306438454154.
[I 2026-05-21 08:02:43,665] Trial 1 finished with value: 0.724107484034235 and parameters: {'learning_rate': 0.04673940788418543, 'max_depth': 6, 'n_estimators': 118, 'min_child_weight': 4, 'subsample': 0.6821678019408679, 'colsample_bytree': 0.9392401426241382}. Best is trial 1 with value: 0.724107484034235.
[I 2026-05-21 08:03:35,289] Trial 2 finished with value: 0.7269283990706249 and parameters: {'learning_rate': 0.08242320805260846, 'max_depth': 3, 'n_estimators': 363, 'min_child_weight': 2, 'subsample': 0.6839341869060721, 'colsample_bytree': 0.9340160658741817

最佳AUC
0.7299078517881131

最佳参数
{'learning_rate': 0.08504153083123973, 'max_depth': 5, 'n_estimators': 449, 'min_child_weight': 5, 'subsample': 0.7539869298815802, 'colsample_bytree': 0.7183081300410319}
